### Entire transformer Block code

In [119]:
import torch
import torch.nn as nn

In [120]:
cfg={
    'emb_dim':768,
    'context_length':1024,
    'num_heads':12,
    'drop_rate':0.1,
    'qkv_bias':False
}

In [121]:
class GELU(nn.Module):
    def __init__(self,):
        super().__init__()
    
    def forward(self,x):
        return 0.5*x*(1+torch.tanh(
            torch.sqrt(torch.tensor(2.0/torch.pi))
            (x+0.044715*torch.pow(x,3))))

In [122]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers=nn.Sequential(
            nn.Linear(cfg['emb_dim'],4*cfg['emb_dim']),
            GELU(),
            nn.Linear(4*cfg['emb_dim'],cfg['emb_dim'])
        )
    
    def forward(self,x):
        return self.layers(x)


In [123]:
torch.ones(cfg['context_length'],cfg['emb_dim'])

tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.]])

In [124]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift
        

In [125]:
torch.rand(cfg['emb_dim'],cfg['emb_dim'])

tensor([[0.8616, 0.5823, 0.3078,  ..., 0.1586, 0.8040, 0.4549],
        [0.7017, 0.6820, 0.3124,  ..., 0.8350, 0.0169, 0.6360],
        [0.2343, 0.3684, 0.6549,  ..., 0.4782, 0.4082, 0.4596],
        ...,
        [0.5596, 0.2183, 0.0666,  ..., 0.1748, 0.4873, 0.0525],
        [0.4352, 0.0656, 0.0195,  ..., 0.1111, 0.2316, 0.0957],
        [0.4762, 0.5958, 0.2140,  ..., 0.3471, 0.5331, 0.8779]])

In [126]:
torch.rand(cfg['emb_dim'],cfg['emb_dim']).shape

torch.Size([768, 768])

In [127]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.att=MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"],
            num_heads=cfg["num_heads"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff=FeedForward(cfg)
        self.layernorm1=LayerNorm(cfg['emb_dim'])
        self.layernorm2=LayerNorm(cfg['emb_dim'])
        self.drop_shortcut=nn.Dropout(cfg['drop_rate'])

    def forward(self,x):
        x_sortcut=x ## 1st shortcut
        x=self.layernorm1(x)
        x=self.att(x)
        x=self.drop_shortcut(x)
        x=x+x_sortcut

        x_sortcut=x  ## 2nd shortcut

        x=self.layernorm2(x)
        x=self.ff(x)
        x=self.drop_shortcut(x)
        x=x+x_sortcut

        return x



In [129]:
torch.manual_seed(123)
x = torch.rand(2, 4, 768) #A
block = TransformerBlock(cfg)
output = block(x)
print("Input shape:", x.shape)
print("Output shape:", output.shape)

TypeError: LayerNorm.forward() missing 1 required positional argument: 'x'

In [ ]:
cfg['emb_dim']

768

In [ ]:
cfg["emb_dim"]

768

In [ ]:
cfg["num_heads"]

12